In [ ]:
%pip -q install rasterio numpy pandas scipy pyproj earthaccess pystac-client

In [ ]:
from pathlib import Path
from google.colab import files

entrada = Path('/content/space_analogs_output_v1_3.zip')
if not entrada.exists():
    print('Envie o ZIP produzido na execução V1.3: space_analogs_output_v1_3.zip')
    arquivos = files.upload()
    nome_zip = next((nome for nome in arquivos if nome.lower().endswith('.zip')), None)
    if nome_zip is None:
        raise FileNotFoundError('Nenhum arquivo ZIP foi enviado.')
    origem = Path('/content') / nome_zip
    if origem != entrada:
        origem.rename(entrada)
print('Entrada encontrada:', entrada, entrada.stat().st_size, 'bytes')


In [ ]:
"""NASA Space Apps / Space Analogs — lote de 43 locais terrestres + Marte/Lua.

Protocolo V1.3.1 exploratório: 9x9 km / 40 m (225x225), gaussian sigma=1 px,
rugosidade STD local 5x5, borda de 3px. Não misture os scores V1.2 (30m)
com os V1.3.1 (40m). Usa NASADEM + ArcticDEM + REMA e GeoTIFFs planetários.

Execução no Colab: veja o notebook companheiro para instalação e upload.
"""
from __future__ import annotations

import json
import math
import re
import shutil
import unicodedata
import zipfile
from contextlib import ExitStack
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.transform import from_origin
from rasterio.warp import reproject, transform_bounds
from rasterio.windows import from_bounds
from scipy.ndimage import gaussian_filter, uniform_filter, distance_transform_edt, binary_dilation

# ==============================================================
# CONFIGURAÇÕES — padrão: processar todos os 43, sem extras.
# ==============================================================
BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
CSV_LOCAIS = BASE / 'terra_por_local.csv'
OUT = BASE / 'space_analogs_output_v1_3_1'
CACHE_NASADEM = BASE / 'nasadem'

RES_M = 40                          # >=32m do DEM polar, evita upsampling nominal
N = 225                             # 225 x 40 = 9.000 metros
SIGMA_PX = 1.0
RUGOSIDADE_JANELA = 5
BORDA_PX = 3
RAIO_EXCLUSAO_NODATA = 7          # raio efetivo dos filtros locais (Gauss 4 + STD 2 + gradiente 1)
MIN_VALID_PCT = 99.9              # aceita no máximo 0,1% de lacunas; registra e exclui seu entorno
SALVAR_RECORTE_TIF = True           # 225x225 / ~200 kB cada, compressão DEFLATE
BAIXAR_NASADEM = True
PROCESSAR_PLANETAS = True

STAC_PGC = 'https://stac.pgc.umn.edu/api/v1/'
PGC_ARCTIC = 'arcticdem-mosaics-v4.1-32m'
PGC_REMA = 'rema-mosaics-v2.0-32m'

FEATURES = [
    'mean_slope_deg', 'roughness_m', 'relative_relief_m',
    'median_slope_deg', 'p90_slope_deg', 'median_roughness_m',
    'p90_roughness_m', 'robust_relief_p95_p5_m'
]
FEATURES_CLUSTER = ['median_slope_deg', 'median_roughness_m', 'robust_relief_p95_p5_m']

# Comparações exploratórias: constantes de tolerância NÃO validadas.
METODOS = {
    'medias_extremos': (['mean_slope_deg', 'roughness_m', 'relative_relief_m'], [2, 2, 150]),
    'medianas_robusto': (FEATURES_CLUSTER, [2, 2, 150]),
    'p90_robusto': (['p90_slope_deg', 'median_roughness_m', 'robust_relief_p95_p5_m'], [5, 2, 150]),
    'sem_rugosidade': (['p90_slope_deg', 'robust_relief_p95_p5_m'], [5, 150]),
}


def slug(txt):
    txt = unicodedata.normalize('NFKD', str(txt)).encode('ascii', 'ignore').decode('ascii')
    return re.sub('[^a-z0-9]+', '_', txt.lower()).strip('_')


def utm_epsg(lat, lon):
    zona = min(60, max(1, math.floor((lon + 180) / 6) + 1))
    return f'EPSG:{(32600 if lat >= 0 else 32700) + zona}'


def escolher_fonte(lat, lon):
    # Para evitar janela cortando limite de NASADEM, use PGC perto das bordas.
    if lat >= 60.0:
        return 'ArcticDEM', 'EPSG:3413'
    if lat <= -56.0:
        return 'REMA', 'EPSG:3031'
    return 'NASADEM', utm_epsg(lat, lon)


def alvo_grid(lat, lon):
    fonte, crs_dest = escolher_fonte(lat, lon)
    x, y = Transformer.from_crs('EPSG:4326', crs_dest, always_xy=True).transform(lon, lat)
    meia = RES_M * N / 2
    return fonte, crs_dest, x, y, from_origin(x - meia, y + meia, RES_M, RES_M)


def codigo_tile(lat, lon):
    return f"{'N' if lat >= 0 else 'S'}{abs(lat):02d}" + f"{'E' if lon >= 0 else 'W'}{abs(lon):03d}"


def necessarios(bbox):
    a, b, c, d = bbox
    if a > c:
        raise ValueError('Recorte cruza antimeridiano; ajustar rotina de tiles.')
    return sorted({codigo_tile(i,j) for i in range(math.floor(b), math.floor(d - 1e-10)+1)
                                      for j in range(math.floor(a), math.floor(c - 1e-10)+1)})


def extrair_zip_nasadem(arquivo):
    CACHE_NASADEM.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(arquivo) as z:
        for info in z.infolist():
            nome = Path(info.filename).name.upper()
            if not re.fullmatch(r'[NS]\d{2}[EW]\d{3}\.HGT', nome):
                continue
            destino = CACHE_NASADEM / nome
            if not destino.is_file():
                with z.open(info) as fi, destino.open('wb') as fo:
                    shutil.copyfileobj(fi, fo)


def indice_nasadem():
    CACHE_NASADEM.mkdir(parents=True, exist_ok=True)
    # Permite reusar os ZIPs que o usuário já baixou em versões anteriores.
    for pasta in (BASE, CACHE_NASADEM):
        for caminho in pasta.glob('NASADEM_HGT_*.zip'):
            extrair_zip_nasadem(caminho)
    idx = {}
    for p in CACHE_NASADEM.rglob('*'):
        if p.is_file() and re.fullmatch(r'[NS]\d{2}[EW]\d{3}\.hgt',p.name,re.I):
            idx[p.stem.upper()] = p
    return idx


_AUTENTICADO = False
_PGC_CLIENTE = None


def tiles_nasadem(bbox):
    global _AUTENTICADO
    codigos = necessarios(bbox)
    existentes = indice_nasadem()
    faltam = [c for c in codigos if c not in existentes]
    if faltam and BAIXAR_NASADEM:
        import earthaccess
        if not _AUTENTICADO:
            print('   NASA Earthdata: login para os tiles ainda não disponíveis.')
            earthaccess.login()
            _AUTENTICADO = True
        granulos = earthaccess.search_data(
            short_name='NASADEM_HGT', version='001', bounding_box=bbox, count=-1)
        if not granulos:
            raise FileNotFoundError(f'Earthdata não encontrou NASADEM para {bbox}')
        baixados = earthaccess.download(granulos, local_path=str(CACHE_NASADEM))
        for p in baixados:
            p = Path(p)
            if p.is_file() and p.suffix.lower() == '.zip':
                extrair_zip_nasadem(p)
        existentes = indice_nasadem()
        faltam = [c for c in codigos if c not in existentes]
    if faltam:
        raise FileNotFoundError('Tiles NASADEM em falta: '+', '.join(faltam))
    return [existentes[c] for c in codigos]


def ler_nasadem(lat, lon, dst_crs, dst_tf):
    # Mosaico só da janela necessária; CRS de origem WGS84 geográfico.
    bbox = transform_bounds(dst_crs, 'EPSG:4326',
                            dst_tf.c, dst_tf.f-N*RES_M,
                            dst_tf.c+N*RES_M, dst_tf.f, densify_pts=21)
    tiles = tiles_nasadem(bbox)
    destino = np.full((N,N), np.nan, dtype='float64')
    with ExitStack() as pilha:
        fontes = [pilha.enter_context(rasterio.open(str(p))) for p in tiles]
        mosaico, transf = merge(fontes, bounds=bbox, res=(1/3600,1/3600),
                                nodata=-32768, method='first')
        reproject(source=mosaico[0], destination=destino,
                  src_transform=transf, src_crs='EPSG:4326', src_nodata=-32768,
                  dst_transform=dst_tf, dst_crs=dst_crs, dst_nodata=np.nan,
                  resampling=Resampling.average)
    return destino, 'NASA NASADEM_HGT V001', ';'.join(p.name for p in tiles), ''


def pgc_client():
    global _PGC_CLIENTE
    if _PGC_CLIENTE is None:
        from pystac_client import Client
        _PGC_CLIENTE = Client.open(STAC_PGC)
    return _PGC_CLIENTE


def ler_pgc(lat, lon, dst_crs, dst_tf, fonte):
    # Pesquisa STAC apenas os mosaicos 32m que cruzam a AOI.
    collection = PGC_ARCTIC if fonte == 'ArcticDEM' else PGC_REMA
    bbox = transform_bounds(dst_crs, 'EPSG:4326',
                            dst_tf.c, dst_tf.f-N*RES_M,
                            dst_tf.c+N*RES_M, dst_tf.f, densify_pts=21)
    itens = list(pgc_client().search(collections=[collection], bbox=bbox).items())
    itens = sorted(itens, key=lambda item: item.id)
    if not itens:
        raise FileNotFoundError(f'PGC não retornou mosaicos {collection} para {lat}, {lon}')
    destino = np.full((N,N), np.nan, dtype='float64')
    usados = []
    # Geotiffs cloud-optimized: GDAL consulta somente blocos do recorte via HTTP range.
    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif',
        GDAL_HTTP_MAX_RETRY='3', GDAL_HTTP_RETRY_DELAY='1',
        GDAL_HTTP_TIMEOUT='35',
    ):
        for item in itens:
            if 'dem' not in item.assets:
                continue
            href = item.assets['dem'].href
            if not href.startswith('https://'):
                continue
            temporario = np.full((N,N), np.nan, dtype='float64')
            # Um mosaico COG pode ter vários GB, mas não é baixado inteiro.
            with rasterio.open(href) as src:
                reproject(source=rasterio.band(src,1), destination=temporario,
                          src_transform=src.transform, src_crs=src.crs,
                          src_nodata=src.nodata, dst_transform=dst_tf,
                          dst_crs=dst_crs, dst_nodata=np.nan,
                          resampling=Resampling.average)
            validos = np.isfinite(temporario) & (temporario > -9000)
            inserir = np.isnan(destino) & validos
            destino[inserir] = temporario[inserir]
            if inserir.any():
                usados.append(item.id)
            if np.isfinite(destino).all():
                break
    return destino, f'PGC {collection}', ';'.join(usados), STAC_PGC


def dem_terra(lat, lon):
    fonte, crs, cx, cy, tf = alvo_grid(lat, lon)
    if fonte == 'NASADEM':
        arr, dataset, arquivos, url = ler_nasadem(lat,lon,crs,tf)
    else:
        arr, dataset, arquivos, url = ler_pgc(lat,lon,crs,tf,fonte)
    return arr, tf, crs, {
        'source_dataset':dataset, 'source_files':arquivos,
        'source_catalog_url':url, 'source_native_resolution_m':32 if fonte!='NASADEM' else 30,
        'center_x_m':cx,'center_y_m':cy,'center_lat':lat,'center_lon':lon,
        'crs_analysis':crs,
    }


def procurar_planeta(padrao):
    # Procura apenas os dois GeoTIFFs que o usuário baixou previamente.
    arqs = sorted(BASE.glob(padrao))
    if not arqs:
        arqs = sorted(BASE.rglob(padrao))
    return arqs[0] if arqs else None


def dem_planeta(arquivo):
    with rasterio.open(str(arquivo)) as src:
        if src.crs is None or src.crs.linear_units not in ('metre', 'meter'):
            raise ValueError('DEM planetário não possui projeção linear em metros.')
        cx = (src.bounds.left+src.bounds.right)/2
        cy = (src.bounds.bottom+src.bounds.top)/2
        m = N*RES_M/2
        dst_tf = from_origin(cx-m,cy+m,RES_M,RES_M)
        # Reprojeção/regradeamento dentro do CRS do próprio corpo; não usa WGS84.
        destino = np.full((N,N),np.nan,dtype='float64')
        reproject(source=rasterio.band(src,1),destination=destino,
                  src_transform=src.transform,src_crs=src.crs,
                  src_nodata=src.nodata,dst_transform=dst_tf,
                  dst_crs=src.crs,dst_nodata=np.nan,
                  resampling=Resampling.average)
        return destino,dst_tf,src.crs,{
            'source_files':arquivo.name,'source_native_resolution_m':abs(src.res[0]),
            'crs_analysis':src.crs.to_wkt(),
            'center_x_m':cx,'center_y_m':cy,
            'center_lat':np.nan,'center_lon':np.nan,
        }


def calcular_features(dem):
    """Extrai oito features; não altera o DEM original nem oculta os NoData.

    Até 0,1% de pixels faltantes: preenchimento vizinho apenas para permitir
    filtros numéricos, mas métricas de declividade e rugosidade ignoram 7 pixels
    de vizinhança de cada lacuna. Relevo e quantis usam somente pixels originais.
    """
    if dem.shape != (N, N):
        raise ValueError('Recorte não tem dimensões N × N.')
    dem = np.asarray(dem, dtype='float64')
    validos = np.isfinite(dem)
    n_validos = int(validos.sum())
    n_ausentes = int(validos.size - n_validos)
    cobertura = 100.0 * validos.mean()
    if cobertura < MIN_VALID_PCT - 1e-7:
        raise ValueError(
            f'Cobertura insuficiente: {cobertura:.4f}% '
            f'(mínima {MIN_VALID_PCT}%; ausentes {n_ausentes}/{validos.size})'
        )
    if n_validos == 0:
        raise ValueError('Recorte sem elevações válidas.')

    # Mantém dem inalterado: só interpola temporariamente para viabilizar filtros.
    if n_ausentes:
        indices = distance_transform_edt(
            ~validos, return_distances=False, return_indices=True
        )
        dem_filtro = dem.copy()
        dem_filtro[~validos] = dem[tuple(indices)][~validos]
    else:
        dem_filtro = dem

    z = gaussian_filter(dem_filtro, sigma=SIGMA_PX, mode='reflect')
    gy, gx = np.gradient(z, RES_M, RES_M)
    decliv = np.degrees(np.arctan(np.hypot(gx, gy)))
    ctr = z - z.mean()
    media = uniform_filter(ctr, size=RUGOSIDADE_JANELA, mode='reflect')
    media2 = uniform_filter(ctr**2, size=RUGOSIDADE_JANELA, mode='reflect')
    rug = np.sqrt(np.maximum(media2 - media**2, 0))

    # Máscara geográfica realmente usada para calcular métricas locais.
    area_metricas = np.ones((N, N), dtype=bool)
    if BORDA_PX:
        area_metricas[:BORDA_PX, :] = False
        area_metricas[-BORDA_PX:, :] = False
        area_metricas[:, :BORDA_PX] = False
        area_metricas[:, -BORDA_PX:] = False
    if n_ausentes:
        zona_exclusao = binary_dilation(
            ~validos,
            structure=np.ones((3, 3), dtype=bool),
            iterations=RAIO_EXCLUSAO_NODATA,
        )
        area_metricas &= ~zona_exclusao
    if not area_metricas.any():
        raise ValueError('Sem pixels confiáveis para métricas locais.')

    decl = decliv[area_metricas]
    r = rug[area_metricas]
    elevacoes = dem[validos]
    p05, p95 = np.percentile(elevacoes, [5, 95])

    return {
        'analysis_resolution_m': RES_M,
        'aoi_size_m': N * RES_M,
        'valid_coverage_pct': float(cobertura),
        'nodata_pixels': n_ausentes,
        'local_metrics_coverage_pct': float(100 * area_metricas.mean()),
        'nodata_treatment': 'nearest_for_filter_only_exclude_7px' if n_ausentes else 'none',
        'mean_slope_deg': float(decl.mean()),
        'roughness_m': float(r.mean()),
        'relative_relief_m': float(elevacoes.max() - elevacoes.min()),
        'median_slope_deg': float(np.median(decl)),
        'p90_slope_deg': float(np.percentile(decl, 90)),
        'median_roughness_m': float(np.median(r)),
        'p90_roughness_m': float(np.percentile(r, 90)),
        'p05_elevation_m': float(p05),
        'p95_elevation_m': float(p95),
        'robust_relief_p95_p5_m': float(p95 - p05),
    }


def salvar_tif(arquivo,dem,tf,crs):
    arquivo.parent.mkdir(parents=True,exist_ok=True)
    with rasterio.open(str(arquivo),'w',driver='GTiff',width=N,height=N,
                       count=1,dtype='float32',crs=crs,transform=tf,
                       nodata=-9999,compress='deflate',predictor=3) as dst:
        dst.write(np.where(np.isfinite(dem),dem,-9999).astype('float32'),1)


def analisar_similarity(df):
    terra = df.loc[df.body=='EARTH']
    targets = df.loc[df.body.isin(['MARS','MOON'])]
    rows=[]
    for method,(cols,tau) in METODOS.items():
        for _,t in targets.iterrows():
            for _,e in terra.iterrows():
                scores={f'{c}_score':100*math.exp(-abs(float(t[c])-float(e[c]))/float(v))
                        for c,v in zip(cols,tau)}
                rows.append({'feature_set':method,'earth_id':e.region_id,
                             'earth_name':e.region_name,'earth_tipo':e.tipo,
                             'target_id':t.region_id,'target_name':t.region_name,
                             'target_body':t.body,'final_score':float(np.mean(list(scores.values()))),
                             **scores})
    if not rows:
        return pd.DataFrame()
    rank = pd.DataFrame(rows).sort_values(['feature_set','target_body','final_score'],
                                           ascending=[True,True,False])
    rank['rank_all']=rank.groupby(['feature_set','target_id']).cumcount()+1
    # Os controles permanecem no ranking completo para avaliar falsos análogos.
    rank['rank_candidates']=np.nan
    for (_, _), grupo in rank[rank.earth_tipo!='controle'].groupby(['feature_set','target_id']):
        rank.loc[grupo.index,'rank_candidates']=np.arange(1,len(grupo)+1)
    return rank


def ler_csv(path):
    catalogo=pd.read_csv(path,encoding='utf-8-sig')
    req={'local','tipo','lat','lon'}
    if not req.issubset(set(catalogo.columns)):
        raise ValueError(f'CSV precisa conter colunas {sorted(req)}')
    catalogo['lat']=pd.to_numeric(catalogo.lat,errors='raise')
    catalogo['lon']=pd.to_numeric(catalogo.lon,errors='raise')
    if catalogo[['lat','lon']].isna().any().any():
        raise ValueError('CSV possui lat/lon vazias')
    if not catalogo.lat.between(-90,90).all() or not catalogo.lon.between(-180,180).all():
        raise ValueError('Coordenadas fora de limites terrestres')
    if catalogo.local.duplicated().any():
        raise ValueError('Nomes de locais duplicados; alterar IDs')
    catalogo['region_id']=catalogo.local.map(lambda x:'earth_'+slug(x))
    if catalogo.region_id.duplicated().any():
        raise ValueError('IDs normalizados duplicados; necessário identificador distinto')
    return catalogo


def exportar(OUT,features,status,df_catalogo,stats):
    OUT.mkdir(parents=True,exist_ok=True)
    df_feat=pd.DataFrame(features)
    df_status=pd.DataFrame(status)
    df_feat.to_csv(OUT/'terrain_features_v1_3_1.csv',index=False)
    df_status.to_csv(OUT/'processing_status.csv',index=False)
    df_catalogo.to_csv(OUT/'earth_candidates_climate_original.csv',index=False)
    if not df_feat.empty:
        earth=df_feat[df_feat.body=='EARTH'].copy()
        targets=df_feat[df_feat.body.isin(['MARS','MOON'])].copy()
        earth.to_csv(OUT/'cluster_input_earth_8features.csv',index=False)
        targets.to_csv(OUT/'cluster_input_targets_8features.csv',index=False)
        essentials=['region_id','region_name','body','tipo','center_lat','center_lon','source_dataset']+FEATURES_CLUSTER
        earth[essentials].to_csv(OUT/'cluster_input_earth_3features.csv',index=False)
        targets[essentials].to_csv(OUT/'cluster_input_targets_3features.csv',index=False)
        rank=analisar_similarity(df_feat)
        rank.to_csv(OUT/'similarity_rankings_exploratory.csv',index=False)
        if not rank.empty:
            rank[rank.earth_tipo!='controle'].to_csv(OUT/'similarity_rankings_candidates.csv',index=False)
    else:
        rank=pd.DataFrame()
    completo=(len(df_feat[df_feat.body=='EARTH'])==len(df_catalogo) and
              len(df_feat[df_feat.body.isin(['MARS','MOON'])])==2)
    metadata={
        'protocol':'v1_3_1_40m_9km', 'generated_at_utc':datetime.now(timezone.utc).isoformat(),
        'expected_earth_sites':len(df_catalogo),'earth_sites_ok':sum(d['body']=='EARTH' for d in features),
        'targets_ok':sum(d['body'] in ('MARS','MOON') for d in features),
        'complete_for_team_delivery':bool(completo),
        'analysis_grid_m':RES_M,'aoi_size_m':N*RES_M,
        'source_native_resolution':{'NASADEM':'~30m','ArcticDEM':'32m','REMA':'32m','Mars_CTX':'20m','Moon_LOLA':'5m'},
        'algorithm':{'gaussian_sigma_px':SIGMA_PX,'roughness_window_px':RUGOSIDADE_JANELA,
                     'edge_exclusion_px':BORDA_PX, 'nodata_neighborhood_exclusion_px':RAIO_EXCLUSAO_NODATA,
                     'min_valid_coverage_pct':MIN_VALID_PCT,'slope':'np.gradient on Gaussian-filtered DEM',
                     'roughness':'local standard deviation on Gaussian-filtered mean-centered DEM',
                     'robust_relief':'DEM percentile95 minus percentile5'},
        'experimental_rank_formula':'100*exp(-abs(x-y)/tolerance), equal weights',
        'experimental_methods':METODOS,
        'limitations':[
            '40m V1.3.1: não comparar scores numericamente com a versão V1.2 de 30m.',
            'A mesma grade de 40m não implica mesma resolução efetiva/incerteza dos DEMs.',
            'ArcticDEM/REMA são modelos de superfície (DSM), podendo incluir vegetação e estruturas.',
            'NASA DEM/DEM polar têm origens e períodos distintos; documentar diferenças.',
            'Recortes planetários centrais são testes, não áreas selecionadas para missões.',
            'Métodos e tolerâncias de similaridade são exploratórios e não demonstram habitabilidade.',
            'Cada local terrestre representado por um recorte de 9x9 km ao redor de ponto informado.',
            'Clima do CSV não entrou na comparação topográfica; procedência temporal não confirmada.',
            'Uma única posição pode não representar regiões naturais extensas.',
            'Até 0,1% de pixels NoData: vizinho próximo só para filtros; zona de 7 px excluída das métricas locais.',
            'Percentis e extremos são calculados somente com elevações originalmente válidas.',
        ],
        'source_links':{
            'NASADEM':'https://search.earthdata.nasa.gov/',
            'PGC_STAC':'https://stac.pgc.umn.edu/api/v1/',
            'PGC_DOCS':'https://www.pgc.umn.edu/guides/stereo-derived-elevation-models/pgc-dem-products-arcticdem-rema-and-earthdem/',
        },
        'runtime':stats,
    }
    (OUT/'processing_metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
    (OUT/'LEIA_ME.txt').write_text(
        'SPACE ANALOGS — EXPORTAÇÃO V1.3.1\n'
        'As features são valores físicos SEM normalização.\n'
        'cluster_input_earth_3features.csv: locais terrestres para clustering.\n'
        'cluster_input_targets_3features.csv: referências Marte/Lua (não usar como treino\n'
        'de K-Means caso o objetivo seja agrupar apenas regiões terrestres).\n'
        'terrain_features_v1_3_1.csv: todas as 8 features por região.\n'
        'processing_status.csv: conferência de regiões faltantes/erro.\n'
        'processing_metadata.json: parâmetros, fontes e alerta de completude.\n'
        'dem_crops/*.tif: recortes numéricos georreferenciados (NoData originais preservados).\n'
        'similarity_rankings_exploratory.csv: ranks experimentais, não probabilidades.\n'
        'ATENÇÃO: somente enviar à equipe como base completa se earth_sites_ok=43 e targets_ok=2.\n',
        encoding='utf-8'
    )
    return df_feat,df_status,rank,completo


def executar(csv_path=CSV_LOCAIS, output_dir=OUT, planetas=True, relatorio=True):
    output_dir=Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)
    # Evita distribuir recortes antigos quando uma nova execução falhar.
    crops_dir = output_dir / 'dem_crops'
    if crops_dir.is_dir():
        for antigo in crops_dir.glob('*.tif'):
            antigo.unlink()
    df_catalogo=ler_csv(csv_path)
    print(f'CSV: {len(df_catalogo)} locais; planejado: '+
          ', '.join(f'{k}={sum(escolher_fonte(r.lat,r.lon)[0]==k for _,r in df_catalogo.iterrows())}'
                    for k in ('NASADEM','ArcticDEM','REMA')))
    features=[];status=[]
    for n,(_,registro) in enumerate(df_catalogo.iterrows(),1):
        ident=registro.region_id; nome=registro.local
        fonte,_=escolher_fonte(float(registro.lat),float(registro.lon))
        print(f'[{n:02d}/{len(df_catalogo):02d}] {nome} ({fonte})',flush=True)
        try:
            dem,tf,crs,meta=dem_terra(float(registro.lat),float(registro.lon))
            feat=calcular_features(dem)
            features.append({'region_id':ident,'region_name':nome,'body':'EARTH',
                             'tipo':registro.tipo,**meta,**feat})
            if SALVAR_RECORTE_TIF:
                salvar_tif(output_dir/'dem_crops'/f'{ident}.tif',dem,tf,crs)
            status.append({'region_id':ident,'name':nome,'source':fonte,'status':'OK',
                           'detail':(f"AVISO: {feat['nodata_pixels']} pixels NoData; "
                                     f"cobertura={feat['valid_coverage_pct']:.4f}%; zona excluída dos filtros")
                           if feat['nodata_pixels'] else ''})
            print(f"   OK inclinação={feat['mean_slope_deg']:.2f}° "
                  f"rugosidade={feat['roughness_m']:.2f}m",flush=True)
        except Exception as e:
            status.append({'region_id':ident,'name':nome,'source':fonte,
                           'status':'ERRO','detail':f'{type(e).__name__}: {e}'})
            print('   ERRO:',type(e).__name__,str(e)[:220],flush=True)
        # Checkpoint a cada local: interrupções não apagam resultados parciais.
        pd.DataFrame(features).to_csv(output_dir/'terrain_features_checkpoint.csv',index=False)
        pd.DataFrame(status).to_csv(output_dir/'processing_status_checkpoint.csv',index=False)
    if planetas:
        planeta_cfg=[
            ('mars_001','Jezero — recorte central','MARS','*M20_JezeroCrater_CTXDEM_20m*.tif',
             'USGS Mars 2020 CTX DEM 20m'),
            ('moon_001','Connecting Ridge — recorte central','MOON','*Site01_final_adj_5mpp_surf*.tif',
             'NASA LOLA Site01 DEM 5m')]
        for ident,nome,corpo,pattern,source in planeta_cfg:
            print(f'[ALVO] {nome}',flush=True)
            try:
                path=procurar_planeta(pattern)
                if path is None:
                    raise FileNotFoundError(f'Envie previamente o GeoTIFF {pattern} ao Colab')
                dem,tf,crs,meta=dem_planeta(path)
                feat=calcular_features(dem)
                features.append({'region_id':ident,'region_name':nome,'body':corpo,
                                 'tipo':'alvo',**meta,'source_dataset':source,**feat})
                if SALVAR_RECORTE_TIF:
                    salvar_tif(output_dir/'dem_crops'/f'{ident}.tif',dem,tf,crs)
                status.append({'region_id':ident,'name':nome,'source':source,'status':'OK','detail':''})
                print('   OK',flush=True)
            except Exception as e:
                status.append({'region_id':ident,'name':nome,'source':source,
                               'status':'ERRO','detail':f'{type(e).__name__}: {e}'})
                print('   ERRO',e,flush=True)
    stats={'processed':len(status),'ok':sum(s['status']=='OK' for s in status),
           'errors':sum(s['status']=='ERRO' for s in status)}
    df_feat,df_status,rank,completo=exportar(output_dir,features,status,df_catalogo,stats)
    arquivo_zip=shutil.make_archive(str(output_dir),'zip',str(output_dir))
    print('\n'+'='*55)
    print(f'RESULTADO: {sum(df_feat.body=="EARTH")}/{len(df_catalogo)} locais terrestres; '
          f'{sum(df_feat.body.isin(["MARS","MOON"]))}/2 alvos planetários')
    print('PACOTE COMPLETO PARA EQUIPE:', 'SIM' if completo else 'NÃO — conferir processing_status.csv')
    print('ARQUIVOS:',output_dir)
    print('ZIP:',arquivo_zip)
    if relatorio:
        print('\nSTATUS COM ERROS:')
        print(df_status[df_status.status=='ERRO'].to_string(index=False))
        if not rank.empty:
            print('\nTOP 5 (conjunto medianas_robusto, incluindo controles para diagnóstico):')
            print(rank[rank.feature_set=='medianas_robusto'].groupby('target_body').head(5)[
                ['target_body','earth_name','earth_tipo','final_score']].to_string(index=False))
    return df_feat,df_status,rank,arquivo_zip


# ===================== RECUPERAÇÃO INCREMENTAL V1.3.1 =====================
# Execute após carregar o ZIP V1.3, sem precisar repetir os 40 locais válidos.
import io
import zipfile
import shutil

ARQUIVO_V13 = BASE / 'space_analogs_output_v1_3.zip'
PASTA_V131 = BASE / 'space_analogs_output_v1_3_1'

def recuperar_v131(zip_entrada=ARQUIVO_V13, pasta_saida=PASTA_V131):
    zip_entrada = Path(zip_entrada)
    pasta_saida = Path(pasta_saida)
    if not zip_entrada.exists():
        raise FileNotFoundError(f'Não encontrei {zip_entrada}. Envie o ZIP V1.3 primeiro.')

    with zipfile.ZipFile(zip_entrada) as z:
        if z.testzip() is not None:
            raise ValueError('O arquivo ZIP original está corrompido.')
        obrigatorios = {
            'terrain_features_v1_3.csv', 'processing_status.csv',
            'earth_candidates_climate_original.csv', 'processing_metadata.json',
        }
        faltam = obrigatorios - set(z.namelist())
        if faltam:
            raise ValueError(f'ZIP original não contém: {sorted(faltam)}')
        if any(Path(n).is_absolute() or '..' in Path(n).parts for n in z.namelist()):
            raise ValueError('ZIP contém caminho não permitido.')
        if pasta_saida.exists():
            shutil.rmtree(pasta_saida)
        pasta_saida.mkdir(parents=True)
        z.extractall(pasta_saida)

    catalogo = ler_csv(pasta_saida / 'earth_candidates_climate_original.csv')
    feat_orig = pd.read_csv(pasta_saida / 'terrain_features_v1_3.csv')
    status_orig = pd.read_csv(pasta_saida / 'processing_status.csv')
    if feat_orig.region_id.duplicated().any():
        raise ValueError('IDs de features duplicados no ZIP de entrada.')

    # Completa novos campos de auditoria dos 42 recortes originais (100% válidos).
    area_interior = (N - 2*BORDA_PX)**2 / (N*N) * 100
    feat_orig['nodata_pixels'] = 0
    feat_orig['local_metrics_coverage_pct'] = area_interior
    feat_orig['nodata_treatment'] = 'none'
    features = feat_orig.to_dict('records')
    status = status_orig.to_dict('records')
    ids_ok = set(feat_orig.region_id)
    pendentes = catalogo.loc[~catalogo.region_id.isin(ids_ok)]
    print(f'ZIP original: {len(ids_ok)} regiões/alvos já processados.')
    print(f'PENDENTES: {len(pendentes)} — ' + ', '.join(pendentes.local.astype(str)))

    for _, reg in pendentes.iterrows():
        ident = reg.region_id
        print(f'Processando somente: {reg.local}', flush=True)
        fonte, _ = escolher_fonte(float(reg.lat), float(reg.lon))
        try:
            dem, tf, crs, meta = dem_terra(float(reg.lat), float(reg.lon))
            feat = calcular_features(dem)
            features.append({
                'region_id': ident, 'region_name': reg.local,
                'body': 'EARTH', 'tipo': reg.tipo,
                **meta, **feat,
            })
            if SALVAR_RECORTE_TIF:
                salvar_tif(pasta_saida/'dem_crops'/f'{ident}.tif', dem, tf, crs)
            # Substitui erro anterior pelo novo status.
            status = [s for s in status if s['region_id'] != ident]
            detalhe = (f"AVISO: {feat['nodata_pixels']} pixels NoData; "
                       f"cobertura={feat['valid_coverage_pct']:.4f}%; zona excluída das métricas locais")
            status.append({'region_id':ident,'name':reg.local,'source':fonte,
                           'status':'OK','detail':detalhe})
            print('  OK', detalhe, flush=True)
        except Exception as e:
            status = [s for s in status if s['region_id'] != ident]
            status.append({'region_id':ident,'name':reg.local,'source':fonte,
                           'status':'ERRO', 'detail':f'{type(e).__name__}: {e}'})
            print('  ERRO:', e, flush=True)

    # Ordena segundo o catálogo para que os arquivos sejam fáceis de conferir.
    ordem = {r.region_id:i for i,r in catalogo.iterrows()}
    features.sort(key=lambda d: ordem.get(d['region_id'], 10_000))
    status.sort(key=lambda d: ordem.get(d['region_id'], 10_000))
    # Mesmos nomes para checkpoint, agora consistentes com a execução final.
    pd.DataFrame(features).to_csv(pasta_saida/'terrain_features_checkpoint.csv', index=False)
    pd.DataFrame(status).to_csv(pasta_saida/'processing_status_checkpoint.csv', index=False)

    stats={'processed':len(status), 'ok':sum(s['status']=='OK' for s in status),
           'errors':sum(s['status']=='ERRO' for s in status)}
    df_feat, df_status, rank, completo = exportar(
        pasta_saida, features, status, catalogo, stats
    )
    arquivo_zip = shutil.make_archive(str(pasta_saida),'zip',str(pasta_saida))
    n_terra = int((df_feat.body=='EARTH').sum())
    n_target = int(df_feat.body.isin(['MARS','MOON']).sum())
    print(f'\nRESULTADO: {n_terra}/43 regiões terrestres; {n_target}/2 alvos.')
    print('Pronto para entrega dos dados físicos:', 'SIM' if completo else 'AINDA NÃO')
    print('ZIP final:', arquivo_zip)
    return df_feat, df_status, rank, arquivo_zip

resultado_v131 = recuperar_v131()

In [ ]:
from google.colab import files
files.download(resultado_v131[3])